In [ ]:
!pip install underthesea

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from transformers import Blip2Processor, Blip2Model, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
from torch.cuda.amp import autocast, GradScaler
from underthesea import word_tokenize

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CẤU HÌNH (CONFIG)
# ==============================================================================
CONFIG = {
    # File dữ liệu huấn luyện gốc (sẽ được tách thành Train + Val)
    "train_csv": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/ViVQA-main/ViVQA-main/train.csv",
    
    # File Test (Hold-out), chỉ dùng để kiểm tra sau cùng, KHÔNG đưa vào train/val
    "test_csv": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/ViVQA-main/ViVQA-main/test.csv",
    
    "train_img_dir": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/train",
    "test_img_dir": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/test",
    
    # File mapping loại câu hỏi (đặc trưng V4)
    "type_mapping_csv": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/answer_type_mapping.csv", 
    
    "save_path": "phobert_blip2_hierarchical_v4_fixed.pth",
    
    "blip_model": "Salesforce/blip2-opt-2.7b", 
    "text_model": "vinai/phobert-base",      
    
    "batch_size": 32,
    "epochs": 25,
    "lr": 5e-5,
    "weight_decay": 0.05,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "max_length": 50,
    "patience": 5,
    "lambda_type": 0.5,

    # Cách 1: Weighted per-sample answer loss theo q_type
    # Model bị phạt nặng hơn khi đoán sai sample thuộc type 1
    "type_loss_weights": {0: 1.0, 1: 3.0, 2: 1.0, 3: 1.0}
}

# ==============================================================================
# 2. HÀM XỬ LÝ DỮ LIỆU & MASKING
# ==============================================================================
def preprocess_vietnamese(text):
    text = str(text).lower().strip()
    return word_tokenize(text, format="text")

def find_image_path(folder, img_id):
    valid_exts = ['.jpg', '.png', '.jpeg']
    img_id_str = str(img_id)
    if img_id_str.lower().endswith(tuple(valid_exts)):
        path = os.path.join(folder, img_id_str)
        if os.path.exists(path): return path
    for ext in valid_exts:
        path = os.path.join(folder, f"{img_id_str}{ext}")
        if os.path.exists(path): return path
    return None

def create_answer_type_mask(csv_path, label_encoder, num_q_types):
    """
    Tạo ma trận mask [num_q_types, num_classes].
    - Giá trị 0: Cho phép (Allow)
    - Giá trị -inf: Chặn (Block)
    """
    print(f"Loading mask from {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error reading mapping file: {e}")
        print("Creating a dummy mask (Allow all) to prevent crash.")
        return torch.zeros((num_q_types, len(label_encoder.classes_)))

    num_classes = len(label_encoder.classes_)
    # Mặc định chặn tất cả (-inf)
    mask = torch.full((num_q_types, num_classes), float('-inf'))
    
    count_mapped = 0
    # Duyệt file mapping để mở khóa
    for _, row in df.iterrows():
        ans_text = str(row['answer']).lower().strip()
        try:
            # Chấp nhận cả tên cột '# type' hoặc 'type'
            q_type = int(row.get('# type', row.get('type', 0)))
        except:
            continue
            
        if ans_text in label_encoder.classes_:
            idx = label_encoder.transform([ans_text])[0]
            if 0 <= q_type < num_q_types:
                mask[q_type, idx] = 0.0
                count_mapped += 1
            
    # SAFETY NET: Nếu một loại câu hỏi không có đáp án nào được map, 
    # ta mở khóa toàn bộ cho loại đó để tránh lỗi NaN.
    for i in range(num_q_types):
        if torch.max(mask[i]) == float('-inf'):
            print(f"Warning: Type {i} has no mapped answers. Unmasking all for this type.")
            mask[i] = 0.0
            
    print(f"Mask created. Mapped {count_mapped} rules.")
    return mask

class ViVQADataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer, label_encoder, unk_token_id):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.unk_token_id = unk_token_id
        self.known_classes = set(label_encoder.classes_)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Xử lý ảnh
        img_id = row.get('img_id', row.get('image'))
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
            
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
        
        # Xử lý câu hỏi
        question = preprocess_vietnamese(row['question'])
        text_encoding = self.tokenizer(
            question, 
            return_tensors="pt", 
            padding="max_length",
            truncation=True,
            max_length=CONFIG['max_length'],
            add_special_tokens=True
        )
        
        input_ids = text_encoding['input_ids'].squeeze(0)
        attention_mask = text_encoding['attention_mask'].squeeze(0)

        # Xử lý nhãn
        answer = str(row['answer']).lower().strip()
        if answer in self.known_classes:
            label = self.label_encoder.transform([answer])[0]
        else:
            label = self.unk_token_id
        
        # Nhãn loại câu hỏi
        q_type = torch.tensor(int(row.get('type', 0)), dtype=torch.long)
            
        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "q_type": q_type,
            "labels": torch.tensor(label, dtype=torch.long)
        }

# ==============================================================================
# 3. KIẾN TRÚC MÔ HÌNH HIERARCHICAL (V4)
# ==============================================================================
class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim, text_dim, embed_dim, num_heads=8):
        super().__init__()
        self.vis_project = nn.Linear(visual_dim, embed_dim)
        self.txt_project = nn.Linear(text_dim, embed_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, visual_features, text_features):
        v_proj = self.vis_project(visual_features)
        t_proj = self.txt_project(text_features)
        
        attn_output, _ = self.multihead_attn(query=t_proj, key=v_proj, value=v_proj)
        combined = self.norm(attn_output + t_proj)
        
        avg_pool = combined.mean(dim=1)
        max_pool = combined.max(dim=1)[0]
        return avg_pool + max_pool

class PhoBERT_BLIP2_VQA_Hierarchical(nn.Module):
    def __init__(self, num_classes, num_q_types, answer_type_mask):
        super().__init__()
        # Image Encoder
        self.blip = Blip2Model.from_pretrained(CONFIG['blip_model'], torch_dtype=torch.float16)
        for param in self.blip.parameters(): param.requires_grad = False
        
        # Text Encoder
        self.phobert = AutoModel.from_pretrained(CONFIG['text_model'])
        
        # Fusion
        embed_dim = 768
        self.fusion = CrossAttentionFusion(visual_dim=1408, text_dim=768, embed_dim=embed_dim)
        
        # Heads
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
        
        self.type_classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_q_types)
        )
        
        # Đăng ký Mask (buffer để không tính gradient nhưng vẫn lưu trong state_dict)
        self.register_buffer('type_mask', answer_type_mask)

    def forward(self, pixel_values, input_ids, attention_mask, target_q_type=None):
        # Vision
        with torch.no_grad():
            vision_outputs = self.blip.vision_model(pixel_values=pixel_values)
            visual_feats = vision_outputs.last_hidden_state
        
        # Text
        text_outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = text_outputs.last_hidden_state
        
        # Fusion
        fused_vec = self.fusion(visual_feats.to(text_feats.dtype), text_feats)
        
        # Logits
        type_logits = self.type_classifier(fused_vec)
        raw_ans_logits = self.classifier(fused_vec)
        
        # --- HIERARCHICAL MASKING ---
        if self.training and target_q_type is not None:
            # Training: Dùng loại câu hỏi thật (Ground Truth) để chọn mask
            indices = target_q_type
        else:
            # Inference/Validation: Dùng loại câu hỏi do model dự đoán
            indices = torch.argmax(type_logits, dim=1)
            
        batch_mask = self.type_mask[indices]
        
        # Cộng mask vào logits
        masked_ans_logits = raw_ans_logits + batch_mask
        
        return masked_ans_logits, type_logits

# ==============================================================================
# 4. TIỆN ÍCH
# ==============================================================================
class EarlyStopping:
    def __init__(self, patience=5, delta=0, path='checkpoint.pth'):
        self.patience, self.delta, self.path = patience, delta, path
        self.counter, self.best_score, self.early_stop = 0, None, False

    def __call__(self, val_acc, model):
        if self.best_score is None:
            self.best_score = val_acc
            self.save_checkpoint(model)
        elif val_acc < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = val_acc
            self.save_checkpoint(model)
            self.counter = 0
    
    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

# ==============================================================================
# 5. VÒNG LẶP HUẤN LUYỆN (CÁCH 1: WEIGHTED PER-SAMPLE LOSS)
# ==============================================================================
def train():
    # Setup
    print(f"Device: {CONFIG['device']}")
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)
    
    # --- LOAD VÀ SPLIT DỮ LIỆU ---
    print(">>> Đang tải dữ liệu Train gốc...")
    full_train_df = pd.read_csv(CONFIG['train_csv'])
    
    # Tách 10% tập train gốc làm tập Validation chuẩn
    # Stratify theo 'type' để cân bằng loại câu hỏi
    try:
        train_df, val_df = train_test_split(
            full_train_df, 
            test_size=0.1, 
            random_state=42, 
            stratify=full_train_df['type']
        )
    except:
        train_df, val_df = train_test_split(full_train_df, test_size=0.1, random_state=42)
        
    print(f"Train size: {len(train_df)} | Validation size (Split): {len(val_df)}")
    
    # --- Encode Answers (Trên toàn bộ tập gốc để đủ class) ---
    all_answers = full_train_df['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    
    # --- Dataloaders ---
    # Train Loader
    train_loader = DataLoader(
        ViVQADataset(train_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id), 
        batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True
    )
    # Val Loader (Dùng ảnh từ train_img_dir vì tách từ train ra)
    val_loader = DataLoader(
        ViVQADataset(val_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id), 
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True
    )
    
    # --- TẠO MASK (V4 Feature) ---
    num_q_types = int(full_train_df['type'].max() + 1)
    type_mask_tensor = create_answer_type_mask(CONFIG['type_mapping_csv'], label_encoder, num_q_types)
    type_mask_tensor = type_mask_tensor.to(CONFIG['device'])
    
    # Init Model
    print("Initializing Hierarchical Model V4...")
    model = PhoBERT_BLIP2_VQA_Hierarchical(
        len(label_encoder.classes_), 
        num_q_types, 
        type_mask_tensor
    ).to(CONFIG['device'])
    
    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * CONFIG['epochs']), num_training_steps=len(train_loader) * CONFIG['epochs'])
    
    # Cách 1: CrossEntropyLoss với reduction='none' để weighted per-sample
    criterion_ans = nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1)
    criterion_type = nn.CrossEntropyLoss()
    
    # Tạo weight lookup tensor cho nhanh (tránh loop Python mỗi batch)
    type_weight_tensor = torch.ones(num_q_types, device=CONFIG['device'])
    for t, w in CONFIG['type_loss_weights'].items():
        if t < num_q_types:
            type_weight_tensor[t] = w
    
    scaler = GradScaler()
    early_stopping = EarlyStopping(patience=CONFIG['patience'], path=CONFIG['save_path'])

    print(f"Type loss weights: {CONFIG['type_loss_weights']}")
    print("Bắt đầu huấn luyện...")
    for epoch in range(CONFIG['epochs']):
        # --- TRAIN ---
        model.train()
        train_ans_correct, train_type_correct, train_total = 0, 0, 0
        train_type_ans_correct = {t: 0 for t in range(num_q_types)}
        train_type_ans_total = {t: 0 for t in range(num_q_types)}
        train_loss_avg = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            q_type_labels = batch['q_type'].to(CONFIG['device'])
            
            optimizer.zero_grad()
            with autocast():
                ans_logits, type_logits = model(pixel_values, input_ids, attention_mask, target_q_type=q_type_labels)
                
                # Cách 1: Per-sample loss, nhân trọng số theo q_type
                per_sample_loss = criterion_ans(ans_logits, labels)           # [batch_size]
                sample_weights = type_weight_tensor[q_type_labels]             # [batch_size]
                loss_ans = (per_sample_loss * sample_weights).mean()
                
                loss_type = criterion_type(type_logits, q_type_labels)
                total_loss = loss_ans + (CONFIG['lambda_type'] * loss_type)
            
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            train_loss_avg += total_loss.item()
            preds = ans_logits.argmax(1)
            train_ans_correct += (preds == labels).sum().item()
            train_type_correct += (type_logits.argmax(1) == q_type_labels).sum().item()
            train_total += labels.size(0)
            
            for t in range(num_q_types):
                mask = (q_type_labels == t)
                if mask.sum() > 0:
                    train_type_ans_correct[t] += (preds[mask] == labels[mask]).sum().item()
                    train_type_ans_total[t] += mask.sum().item()
            
            pbar.set_postfix({'Loss': f'{total_loss.item():.2f}'})

        # --- VALIDATION ---
        model.eval()
        val_ans_correct, val_type_correct, val_total = 0, 0, 0
        val_type_ans_correct = {t: 0 for t in range(num_q_types)}
        val_type_ans_total = {t: 0 for t in range(num_q_types)}
        
        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                q_type_labels = batch['q_type'].to(CONFIG['device'])
                
                ans_logits, type_logits = model(pixel_values, input_ids, attention_mask)
                
                preds = ans_logits.argmax(1)
                val_ans_correct += (preds == labels).sum().item()
                val_type_correct += (type_logits.argmax(1) == q_type_labels).sum().item()
                val_total += labels.size(0)
                
                for t in range(num_q_types):
                    mask = (q_type_labels == t)
                    if mask.sum() > 0:
                        val_type_ans_correct[t] += (preds[mask] == labels[mask]).sum().item()
                        val_type_ans_total[t] += mask.sum().item()
        
        val_ans_acc = val_ans_correct / val_total
        val_type_acc = val_type_correct / val_total
        
        print(f"\n>>> Epoch {epoch+1} Summary:")
        print(f"    Train Loss: {train_loss_avg/len(train_loader):.4f}")
        print(f"    Train Ans Acc: {train_ans_correct/train_total:.4f} | Val Ans Acc: {val_ans_acc:.4f}")
        print(f"    Train Type Acc: {train_type_correct/train_total:.4f} | Val Type Acc: {val_type_acc:.4f}")
        
        print(f"    --- Per Q-Type Answer Accuracy ---")
        for t in range(num_q_types):
            tr_acc = train_type_ans_correct[t] / train_type_ans_total[t] if train_type_ans_total[t] > 0 else 0
            vl_acc = val_type_ans_correct[t] / val_type_ans_total[t] if val_type_ans_total[t] > 0 else 0
            w = CONFIG['type_loss_weights'].get(t, 1.0)
            marker = " <<<" if w > 1.0 else ""
            print(f"    Type {t} (w={w}): Train={tr_acc:.4f} | Val={vl_acc:.4f}{marker}")
        
        # SỬA EARLY STOPPING TẠI ĐÂY
        val_type_1_acc = val_type_ans_correct[1] / val_type_ans_total[1] if val_type_ans_total[1] > 0 else 0
        print(f"    => Đang theo dõi Early Stopping theo q_type=1 Val Acc: {val_type_1_acc:.4f}")
        
        early_stopping(val_type_1_acc, model)
        if early_stopping.early_stop:
            print("Early stopping triggered dựa trên điểm số của q_type=1!")
            break

if __name__ == "__main__":
    train()

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from PIL import Image
import os
import csv
import gc

# ==========================================
# 1. CẤU HÌNH INFERENCE
# ==========================================
INFERENCE_CONFIG = {
    "train_csv": CONFIG['train_csv'],
    "test_csv": CONFIG['test_csv'],
    "img_dir": CONFIG['test_img_dir'],
    "model_path": CONFIG['save_path'],
    "output_file": "ket_qua_test_vivqa.csv",
    "batch_size": 16, 
    "device": CONFIG['device']
}

# ==========================================
# 2. DATASET CHO INFERENCE
# ==========================================
class ViVQAInferenceDataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        img_id = str(row.get('img_id', row.get('image')))
        question_text = row['question']
        gt_answer = str(row['answer'])
        gt_type = int(row.get('type', 0))
        
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

        processed_q = preprocess_vietnamese(question_text)
        text_encoding = self.tokenizer(
            processed_q, 
            return_tensors="pt", 
            padding="max_length", 
            truncation=True, 
            max_length=CONFIG['max_length'],
            add_special_tokens=True
        )
        
        return {
            "pixel_values": pixel_values,
            "input_ids": text_encoding['input_ids'].squeeze(0),
            "attention_mask": text_encoding['attention_mask'].squeeze(0),
            "meta_img_id": img_id,
            "meta_question": question_text,
            "meta_gt_answer": gt_answer,
            "meta_gt_type": gt_type
        }

def run_inference_and_save():
    # --- Dọn dẹp bộ nhớ ---
    print(">>> Đang dọn dẹp bộ nhớ GPU...")
    if 'model' in globals(): del model
    if 'optimizer' in globals(): del optimizer
    if 'scheduler' in globals(): del scheduler
    gc.collect()
    torch.cuda.empty_cache()

    print(">>> Đang chuẩn bị dữ liệu cho Inference V4...")
    
    # 1. Rebuild LabelEncoder (Khớp với tập Train)
    train_df = pd.read_csv(INFERENCE_CONFIG['train_csv'])
    train_answers = train_df['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in train_answers: train_answers.append('unknown')
    
    label_encoder = LabelEncoder().fit(train_answers)
    num_classes = len(label_encoder.classes_)
    
    # 2. Tái tạo Mask (BẮT BUỘC cho V4 Hierarchical)
    num_q_types = int(train_df['type'].max() + 1)
    
    # Cần file mapping để tạo mask giống lúc train
    type_map_path = CONFIG.get('type_mapping_csv', 'answer_type_mapping.csv')
    if os.path.exists(type_map_path):
        type_mask_tensor = create_answer_type_mask(type_map_path, label_encoder, num_q_types)
    else:
        print("!!! Cảnh báo: Không thấy file mapping. Tạo mask rỗng để load weight.")
        # Tạo mask toàn số 0 (không chặn gì cả) để khớp kích thước weight
        type_mask_tensor = torch.zeros((num_q_types, num_classes))
        
    type_mask_tensor = type_mask_tensor.to(INFERENCE_CONFIG['device'])

    # 3. Khởi tạo đúng Model V4
    print(">>> Khởi tạo model PhoBERT_BLIP2_VQA_Hierarchical...")
    # LƯU Ý: Phải dùng class Hierarchical và truyền mask vào
    model = PhoBERT_BLIP2_VQA_Hierarchical(num_classes, num_q_types, type_mask_tensor)
    
    # 4. Load Weights
    if os.path.exists(INFERENCE_CONFIG['model_path']):
        print(f"Loading weights: {INFERENCE_CONFIG['model_path']}")
        state_dict = torch.load(INFERENCE_CONFIG['model_path'], map_location='cpu')
        model.load_state_dict(state_dict)
    else:
        print("!!! LỖI: Không tìm thấy file model checkpoint.")
        return

    model = model.to(INFERENCE_CONFIG['device'])
    model.eval()

    # 5. Dataloader Test
    test_df = pd.read_csv(INFERENCE_CONFIG['test_csv'])
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)
    
    test_dataset = ViVQAInferenceDataset(test_df, INFERENCE_CONFIG['img_dir'], blip_processor, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=INFERENCE_CONFIG['batch_size'], shuffle=False, num_workers=0)
    
    results = []
    print(">>> Bắt đầu chạy dự đoán...")
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            pixel_values = batch['pixel_values'].to(INFERENCE_CONFIG['device'], dtype=torch.float16)
            input_ids = batch['input_ids'].to(INFERENCE_CONFIG['device'])
            attention_mask = batch['attention_mask'].to(INFERENCE_CONFIG['device'])
            
            with torch.cuda.amp.autocast(enabled=(INFERENCE_CONFIG['device']=="cuda")):
                # V4 Inference: Model tự dùng type dự đoán để áp mask
                ans_logits, type_logits = model(pixel_values, input_ids, attention_mask)
            
            pred_ans_indices = ans_logits.argmax(dim=1).cpu().numpy()
            pred_type_indices = type_logits.argmax(dim=1).cpu().numpy()
            pred_ans_text = label_encoder.inverse_transform(pred_ans_indices)
            
            batch_size = input_ids.size(0)
            for i in range(batch_size):
                results.append({
                    "ID ảnh": batch['meta_img_id'][i],
                    "Câu hỏi": batch['meta_question'][i],
                    "Đáp án (GT)": batch['meta_gt_answer'][i],
                    "Q_Type (GT)": batch['meta_gt_type'][i].item(),
                    "Q_Type Dự đoán": pred_type_indices[i],
                    "Đáp án Dự đoán": pred_ans_text[i]
                })

    # 6. Lưu kết quả
    result_df = pd.DataFrame(results)
    cols = ["ID ảnh", "Câu hỏi", "Đáp án (GT)", "Q_Type (GT)", "Q_Type Dự đoán", "Đáp án Dự đoán"]
    result_df = result_df[cols]
    
    result_df.to_csv(INFERENCE_CONFIG['output_file'], index=False, encoding='utf-8-sig')
    print(f"\n>>> Đã lưu kết quả: {INFERENCE_CONFIG['output_file']}")
    
    acc = (result_df["Đáp án (GT)"].astype(str).str.lower().str.strip() == result_df["Đáp án Dự đoán"].astype(str).str.lower().str.strip()).mean()
    print(f"Độ chính xác Test: {acc:.4f}")

if __name__ == "__main__":
    run_inference_and_save()